<a href="https://colab.research.google.com/github/deepakk7195/IISC_CDS_DS/blob/capstone_project_group12/gradio_UI_implementation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# pip install --upgrade gradio

In [4]:
import gradio as gr
import tensorflow as tf
from tensorflow.keras.preprocessing.image import img_to_array
from tensorflow.keras.applications.resnet50 import preprocess_input
import numpy as np
from PIL import Image
import random

In [11]:
import gradio as gr
import random

# Define classes for the first tab (disease classification)
disease_classes = [
    "Normal",
    "Glaucoma",
    "Diabetic Retinopathy",
    "Macular Degeneration",
    "Hypertensive Retinopathy",
    "Cataract",
    "Retinal Detachment",
    "Optic Neuritis"
]

# Define a global variable to store the image from the first tab
shared_image = None

# Dummy model for disease classification
def classify_disease(image, actual_label):
    global shared_image
    shared_image = image  # Save the image for use in the second tab
    if image is None or actual_label not in disease_classes:
        return "Error: Invalid input", "Error: Invalid input"

    predicted_class = random.choice(disease_classes)
    return predicted_class, actual_label

# Dummy model for retinopathy grading
def grade_retinopathy(image, grading_label):
    if image is None or grading_label not in ["0", "1", "2", "3", "4"]:
        return "Error: Invalid input", "Error: Invalid input"

    predicted_grade = random.choice(["0", "1", "2", "3", "4"])
    return predicted_grade, grading_label

# First tab interface (Disease Classification)
tab1 = gr.Interface(
    fn=classify_disease,
    inputs=[
        gr.Image(type="pil", label="Upload Fundus Image"),
        gr.Dropdown(choices=disease_classes, label="Select Actual Disease Label")
    ],
    outputs=[
        gr.Textbox(label="Predicted Class"),
        gr.Textbox(label="Actual Label Selected")
    ],
    title="Disease Classification",
    description="Upload a fundus image and classify it into one of the disease categories."
)

# Second tab interface (Retinopathy Grading)
with gr.Blocks() as tab2:
    with gr.Row():
        img_option = gr.Radio(["Use Same Image", "Upload New Image"], value="Upload New Image", label="Image Source")
    with gr.Row():
        image_input = gr.Image(type="pil", label="Upload Fundus Image (if applicable)", visible=True)
    with gr.Row():
        grade_label = gr.Dropdown(choices=["0", "1", "2", "3", "4"], label="Select Retinopathy Grade")
    with gr.Row():
        predict_button = gr.Button("Classify Retinopathy")
    with gr.Row():
        predicted_grade = gr.Textbox(label="Predicted Grade")
        selected_grade = gr.Textbox(label="Selected Grade")

    # Function to toggle image input visibility
    def toggle_image_input(image_source):
        if image_source == "Use Same Image":
            return gr.update(visible=False)  # Hide input
        return gr.update(visible=True)  # Show input

    img_option.change(toggle_image_input, img_option, image_input)

    # Function to handle retinopathy grading
    def handle_retinopathy(image, grade):
        global shared_image
        if image is None:  # Use shared image if "Use Same Image" is selected
            image = shared_image
        return grade_retinopathy(image, grade)

    predict_button.click(handle_retinopathy, [image_input, grade_label], [predicted_grade, selected_grade])

# Combine the two tabs
demo = gr.TabbedInterface([tab1, tab2], ["Disease Classification", "Retinopathy Grading"])

# Launch the app
demo.launch(share=True)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://aa5edc902dd922608d.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
